# From a company name to files on disk

`find_company` answers the question you have before `Company()`: which market
does this company file in, and which identifier does that market use? Then the
match opens its route, and `file_types` saves just the files you want.

## Identify your SEC requests

The SEC asks automated clients for a name or organization and a contact email,
separated by a space. For example, `Alex Morgan alex@example.com` or
`Example Research filings@example.com`. These are examples: use your own name
and a working email. No SEC account or API key is needed.

In the next cell, uncomment the `os.environ[...]` line and replace the example
with your details. If you already set this environment variable in your
terminal, leave the line commented. The label is sent to the SEC with each
request. See the [setup guide](../../docs/getting-started.md#sec-request-identity).

In [1]:
import os
from collections import Counter

from opencorpfilings import Company, provider_info, search_companies

# Uncomment this line and replace the example with your own name and contact email.
# os.environ["OPENCORPFILINGS_SEC_USER_AGENT"] = "Alex Morgan alex@example.com"
assert os.getenv("OPENCORPFILINGS_SEC_USER_AGENT"), "Uncomment the SEC identity example above and replace it with your own details, or set OPENCORPFILINGS_SEC_USER_AGENT in your environment."

from opencorpfilings import find_company

The first call builds a local index from official lists (SEC tickers, EDINET
listed companies, the AMF's LEI and ISIN lists, the ESEF index by country), in
about 20 seconds. Later calls reuse it for seven days.

In [2]:
find_company("Toyota", limit=4)

[<CompanyMatch US 1094517 'TOYOTA MOTOR CORP/' supported>, <CompanyMatch JP 7203 'TOYOTA MOTOR CORPORATION' supported>, <CompanyMatch JP 8015 'TOYOTA TSUSHO CORPORATION' supported>, <CompanyMatch JP 3116 'TOYOTA BOSHOKU CORPORATION' supported>]

Search by whatever you have: a ticker, a Tokyo securities code, an EDINET code, an ISIN or an LEI.

In [3]:
for query in ["AAPL", "9984", "E02144", "FR0010307819", "5299006UDSEJCTTEJS30"]:
    best = find_company(query, limit=1, live=False)[0]
    print(f"{query:22} -> {best.market} {best.identifier:22} {best.name}")

AAPL                   -> US 320193                 Apple Inc.
9984                   -> JP 9984                   SoftBank Group Corp.
E02144                 -> JP 7203                   TOYOTA MOTOR CORPORATION
FR0010307819           -> FR 969500XXRPGD7HCAFA90   LEGRAND
5299006UDSEJCTTEJS30   -> AT 5299006UDSEJCTTEJS30   VERBUND AG


Each match says where it came from and what you can expect from its route:

In [4]:
match = find_company("Legrand", limit=1)[0]
match.market, match.identifier, match.tier, match.lei, match.isins, match.listed_by

('FR', '969500XXRPGD7HCAFA90', 'supported', '969500XXRPGD7HCAFA90', ('FR0010307819',), 'AMF/DILA codes-lei and liste-code-isi')

The same company can be reachable by more than one route. Supported routes rank
first; the ESEF index is the `limited` alternative with older history.

In [5]:
[(m.market, m.source, m.tier) for m in find_company("Legrand", markets=["FR"], live=False)]

[('FR', None, 'supported'), ('FR', 'esef_index', 'limited')]

The UK has no list of listed companies, so its search runs live at Companies
House and returns any registered company. Those hits rank below a listed issuer
with the same name.

In [6]:
[(m.market, m.identifier, m.name, m.listed_by) for m in find_company("Tesco", limit=3)]

[('UK', '00445790', 'TESCO PLC', 'Companies House search (live)'), ('UK', '05888957', 'TESCO AQUA (FINCO2) LIMITED', 'Companies House search (live)'), ('US', '802257', 'Mitesco, Inc.', 'SEC company_tickers_exchange.json')]

Open a match's route and get its latest annual report:

In [7]:
verbund = find_company("Verbund", limit=1, live=False)[0].company()
filing = verbund.filings(form="annual").latest()
filing, [(document.filename, document.role) for document in filing.documents]

(<Filing 24793 ESEF ANNUAL period=2025-12-31 AT/esef_index>, [('verbundag-2025-12-31-1-en.xhtml', 'report'), ('5299006UDSEJCTTEJS30-2025-12-31.xbri', 'report'), ('verbundag-2025-12-31-1-en.json', 'data')])

`file_types` keeps documents by extension, across all of each filing's
documents, here the XBRL schema and linkbases of Apple's latest 10-K:

In [8]:
apple = find_company("AAPL", limit=1, live=False)[0].company()
result = apple.filings(form="annual", limit=1).download("downloads/apple-taxonomy", file_types=(".xsd", ".xml"))
result

{'manifest': 'downloads\\apple-taxonomy\\manifest.json', 'filings': 1, 'downloaded': 7, 'skipped': 0, 'failed': 0}

In [9]:
import json
manifest = json.load(open(result["manifest"], encoding="utf-8"))
[(document["filename"], document["role"], document["bytes"]) for document in manifest["filings"][0]["documents"]]

[('aapl-20250927.xsd', 'data', 56313), ('aapl-20250927_cal.xml', 'data', 153156), ('aapl-20250927_def.xml', 'data', 224664), ('aapl-20250927_htm.xml', 'data', 1416841), ('aapl-20250927_lab.xml', 'data', 763631), ('aapl-20250927_pre.xml', 'data', 496062), ('FilingSummary.xml', 'generated', 42775)]